# Week 1 · IT Helpdesk Resolution — Dataset Selection, Freeze & Baseline

**Capstone Track A** — domain-adapted assistant (fine-tune + serve a ≤8B open LLM).
This notebook covers the **Week 1 hard gate**: *dataset frozen* and *baseline numbers recorded*.

| Stage | What happens | Output |
|---|---|---|
| 1 | Load the chosen dataset, inspect, filter to IT-style queues | clean dataframe |
| 2 | Suitability checks (size, label balance, length, PII scan + redaction) | evidence for the report |
| 3 | Define the target schema and prompt format | `SYSTEM`, SFT records |
| 4 | Stratified split, TF-IDF leakage guard, **freeze** with SHA-256 manifest | `train/val/test.jsonl`, `manifest.json` |
| 5 | Zero-shot and few-shot baseline of the base model on the held-out test set | `preds_*.jsonl` |
| 6 | Metrics, error analysis, optional LLM-as-judge | `baseline_results.json` |

**Runtime:** any GPU with 15 GB or more (developed on Tesla T4s; the recorded baseline was produced on an NVIDIA A100 40 GB, and the same notebook runs on an L40S). Run All top to bottom. No API keys needed unless you flip `RUN_JUDGE`. By default the last cell shuts the kernel down so the GPU is completely free for Week 2 (`SHUTDOWN_WHEN_DONE`).
Cells up to the dataset freeze need only a CPU; the GPU is first used in the model-loading step.

**How the four notebooks fit together (this is notebook 1 of 4)**

| Order | Notebook | Reads | Writes |
|---|---|---|---|
| 1 | `it_helpdesk_week1_baseline.ipynb` (this one) | the Hugging Face dataset | frozen `train/val/test.jsonl`, `manifest.json`, `preds_*.jsonl`, `baseline_results.json` |
| 2 | `it_helpdesk_week2_qlora.ipynb` | the frozen files above | adapters, `run_*.json`, comparison tables |
| 3 | `it_helpdesk_week3_rag.ipynb` | the frozen files, this notebook's baseline, Week 2's adapters | knowledge base, retrieval decisions, RAG predictions, `week3_results.json` |
| 4 | `it_helpdesk_week4_serving.ipynb` | all of the above | merged and quantised model, FastAPI + vLLM service repo, model card, red-team log, demo kit |

- Each notebook is **self-contained**: it checks its own libraries, verifies what it needs from disk, and stops with a clear message if something is missing. You do not need to keep this notebook's kernel alive for Week 2.
- **GPU memory:** a notebook kernel holds on to GPU memory until it is shut down, even when idle. The last cell of this notebook (*Step 6*) frees the model, and Week 2 begins with a check that refuses to start if the GPU is still occupied. Finish this notebook, run Step 6, then open Week 2. Never run both notebooks' GPU cells at the same time.
- **Frozen data is protected:** once `manifest.json` exists, re-running the freeze step with different settings stops with an error instead of silently overwriting `train/val/test.jsonl` (set `FORCE_REFREEZE = True` only if you really want a new freeze).

**Metrics** (all computed on the same 150–200 held-out tickets):

- *Schema validity*: output parses to a JSON object with the 4 required keys and in-vocabulary labels.
- *Field accuracy / macro-F1* for `queue`, `type`, `priority`: $F1_{macro}=\frac{1}{|C|}\sum_{c\in C}\frac{2P_cR_c}{P_c+R_c}$
- *ROUGE-L F1* of `resolution` against the reference reply (invalid outputs score 0).
- *Throughput*: generated tokens per second on this HF-generate setup (final serving numbers come from vLLM in Week 4).

### Step 0a · Check that the required libraries are installed

**What the next cell does**

It checks that every third-party library this notebook relies on can be imported, prints the versions, and stops with a ready-to-copy install command if anything is missing. Think of it as checking your toolbox before starting the job. We *check* instead of installing from inside the notebook because this project's `.venv` is managed by `uv`, which does not ship `pip`, so the old `%pip install` line could not work here (it printed *No module named pip*).

| Library | Why we need it |
|---|---|
| `torch` | The deep-learning engine everything else runs on |
| `transformers` | Loads the Qwen language model and its tokenizer, and runs text generation |
| `accelerate` | Places the model on the GPU for us (`device_map="auto"`) |
| `bitsandbytes` | Squeezes the 7B model into 4-bit numbers so it fits on a 15 GB T4 GPU |
| `datasets` | Downloads the ticket dataset from the Hugging Face Hub |
| `scikit-learn` | Stratified splitting, TF-IDF similarity and the F1 metric |
| `rouge-score` | ROUGE-L, which measures how close a generated reply is to the reference reply |
| `pandas`, `jinja2` | Tables, plus the coloured table styling used in the results |

**How it works**

1. `PACKAGES` maps the name you install with (`pip install scikit-learn`) to the name you import (`import sklearn`). They differ for a few libraries, which is a classic source of confusion.
2. `importlib.util.find_spec` asks Python whether a module can be found *without actually importing it*, so the check is fast and does not touch the GPU.
3. If anything is missing we print the exact `uv pip install ...` command and raise an error, so "Run All" stops here instead of failing ten cells later. On Colab, where `pip` exists, put `%pip install -q <packages>` in a cell instead.

**After installing anything, restart the kernel**, otherwise Python keeps using the old versions already loaded in memory.

In [1]:
import importlib.util
from importlib.metadata import version, PackageNotFoundError

# pip name -> import name (they differ for scikit-learn and rouge-score)
PACKAGES = {"torch": "torch", "transformers": "transformers", "accelerate": "accelerate",
            "bitsandbytes": "bitsandbytes", "datasets": "datasets", "scikit-learn": "sklearn",
            "rouge-score": "rouge_score", "pandas": "pandas", "jinja2": "jinja2"}

# find_spec() only looks the module up; it does not import it, so this check is quick and safe
missing = [pip for pip, mod in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Missing libraries. In a terminal run:\n    uv pip install --python .venv/bin/python " + " ".join(missing))
    print("(on Colab use:  %pip install -q " + " ".join(missing) + ")")
    raise ImportError(f"missing libraries: {missing}; install them, then restart the kernel")

for pip in PACKAGES:
    try: print(f"{pip:14s} {version(pip)}")
    except PackageNotFoundError: pass

torch          2.9.1+cu126
transformers   5.23.0
accelerate     1.16.0
bitsandbytes   0.50.2
datasets       5.2.0
scikit-learn   1.9.1
rouge-score    0.1.2
pandas         3.1.1
jinja2         3.1.6


### Step 0b · Imports and the control panel (all settings in one place)

**What the next cell does**

Nothing is computed here. This cell is the notebook's *control panel*: it imports the standard libraries and defines every setting the later cells read. Keeping the knobs in one place means you can change an experiment by editing this single cell instead of hunting through the notebook.

**The settings, grouped**

- **Reproducibility.** `SEED = 42` is given to Python's and NumPy's random generators, so the same split and the same few-shot examples come out on every run.
- **Model and generation.** `MODEL_ID` is the base model we are baselining. `LOAD_4BIT = True` loads it in 4-bit (NF4), the same format the QLoRA fine-tuning will use later, which keeps the before/after comparison fair. `MAX_NEW_TOKENS` caps how long an answer can be, and `GEN_BATCH` is how many tickets go through the GPU at once.
- **Dataset and filters.** Which Hugging Face dataset to use, which language, how to recognise IT-style queues (`QUEUE_REGEX`), and the length limits that drop tickets that are too short, too long or empty.
- **Split sizes.** `N_TEST`, `N_VAL`, `N_TRAIN` set the size of the three splits, and `DEDUP_SIM` is the similarity threshold used to remove near-copies of test tickets from the training pool.
- **Baseline switches.** `K_SHOT` is the number of worked examples in the few-shot prompt, and `SMOKE_N` is a quick-test mode (for example `SMOKE_N = 8` runs only 8 tickets). `USE_WANDB` and `RUN_JUDGE` turn the optional experiment tracking and the LLM judge on or off.
- **Safety and housekeeping switches.** `FORCE_REFREEZE` (default `False`) is the guard on the frozen data: with it off, the freeze step refuses to overwrite existing `train/val/test.jsonl` files if the new data would differ from `manifest.json`. `SHUTDOWN_WHEN_DONE` (default `True`) makes the final cleanup cell stop the kernel completely, which is the only way to return *all* GPU memory (see Step 6). That is what makes running Week 1 and then Week 2 back to back safe; set it to `False` if you want to keep the kernel alive to inspect variables.
- **GPU allocator setting.** `expandable_segments:True` (set through the `PYTORCH_ALLOC_CONF` environment variable, plus its older name for older PyTorch versions) is applied *before* PyTorch touches the GPU. It lets the memory allocator grow blocks instead of carving the GPU into fixed chunks, which reduces "out of memory although memory is free" errors caused by fragmentation.
- **Output folder.** Everything we save (splits, manifest, predictions, results) goes to `capstone_it_helpdesk/week1/`.

In [2]:
import os, re, json, time, random, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")   # hide noisy library warnings so the printed results stay readable

# Fix every random generator we use. Same seed -> same split, same few-shot examples, repeatable results.
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- model / generation (keep identical for baseline and fine-tuned evals)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # alt: meta-llama/Llama-3.1-8B-Instruct (gated), mistralai/Mistral-7B-Instruct-v0.3
LOAD_4BIT = True                        # same NF4 base the QLoRA runs will use -> fair comparison
MAX_NEW_TOKENS = 320
GEN_BATCH = 8

# --- dataset
DATASET_ID = "Tobi-Bueck/customer-support-tickets"
LANG = "en"
QUEUE_OVERRIDE = None                   # e.g. ["IT Support", "Technical Support", ...] to bypass auto-selection
# Regex used to spot IT-style queue names: the word "IT" on its own, or any of the keywords (case-insensitive)
QUEUE_REGEX = r"\bIT\b|(?i:tech|network|security|access|outage|software|hardware|infrastructure)"
# Keep between 3 and 8 queues, and ignore queues with fewer than 60 tickets (too few to learn or evaluate on)
MIN_QUEUES, MAX_QUEUES, MIN_PER_QUEUE = 3, 8, 60
# Length filters (in characters) drop tickets/replies that are trivially short or absurdly long
MIN_BODY_CHARS, MAX_BODY_CHARS = 40, 1200
MIN_ANS_CHARS, MAX_ANS_CHARS = 60, 900
# Target split sizes. The test set is the "final exam", so it is kept untouched and the same for every model
N_TEST, N_VAL, N_TRAIN = 180, 100, 1500  # test 150-200; train+val must stay <= 2000
DEDUP_SIM = 0.90                         # TF-IDF cosine above which a train row is dropped as a test near-duplicate

# --- baseline
K_SHOT = 3
SMOKE_N = None                           # int -> evaluate only the first N test rows (quick GPU check)
def _wandb_ready():
    # experiment tracking switches itself on when the `wandb` package is installed AND you are logged in (`wandb login` or WANDB_API_KEY)
    import importlib.util
    if importlib.util.find_spec("wandb") is None:
        return False
    if os.environ.get("WANDB_API_KEY"):
        return True
    try:
        return "api.wandb.ai" in (Path.home() / ".netrc").read_text()
    except OSError:
        return False
USE_WANDB = _wandb_ready()
RUN_JUDGE = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("JUDGE_API_KEY"))   # runs automatically when a judge key is set in the environment
JUDGE_MODEL, JUDGE_N = os.environ.get("JUDGE_MODEL", "claude-sonnet-5-5"), None   # None = judge every test ticket (the brief asks for the held-out set)

# --- safety / housekeeping
FORCE_REFREEZE = False                   # True = allow the freeze step to overwrite existing train/val/test.jsonl
SHUTDOWN_WHEN_DONE = True                # True = the LAST cell stops the kernel so ALL GPU memory is returned (set False to keep the kernel)

# Must be set before PyTorch first touches the GPU: expandable segments reduce memory fragmentation (fewer false OOMs).
# Newer PyTorch reads PYTORCH_ALLOC_CONF; older versions only know PYTORCH_CUDA_ALLOC_CONF, so we set both.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Everything we save lands in this folder; exist_ok=True means re-running the cell won't crash if it already exists
OUT = Path("capstone_it_helpdesk") / "week1"; OUT.mkdir(parents=True, exist_ok=True)

## 1 · Dataset selection: how the dataset was chosen, and what we verified

Choosing the dataset is the single decision that limits everything downstream: a fine-tuned model can only be as good (and as honest) as the data it learns from. This section records the **requirements**, the **candidates**, the **decision** and the **evidence from the data itself**, so that the choice can be defended in the report and the model card.

### 1. Requirements derived from the capstone brief

| # | Requirement | Why it matters |
|---|---|---|
| R1 | **Enough pairs**: 800–2,000 training pairs *plus* 150–200 held-out test pairs, after filtering | The brief fixes the sizes; filtering (language, length, duplicates) removes many rows, so the source must be much larger |
| R2 | **Ticket → resolution pairs** (a problem text *and* an agent reply) | The task is "resolution", so a dataset with tickets only (no answers) cannot be used for supervised fine-tuning |
| R3 | **Objective labels** (queue, type, priority) | They give cheap, deterministic metrics (exact match, macro-F1, schema validity) next to the subjective ones (ROUGE, judge) |
| R4 | **IT-helpdesk domain fit** | The brief asks for one narrow enterprise domain |
| R5 | **Clear licence and provenance** | A *regulated enterprise* workflow needs a data-provenance section in the model card |
| R6 | **Publicly downloadable, reproducible** | Every team member must be able to re-create the data; no paywall, no access request |
| R7 | **English text** | The base models are strongest in English and the metrics (ROUGE) assume word tokens |
| R8 | **Privacy-safe** | No real personal data, or at least a way to detect and redact it |

### 2. Candidates and how they scored

Candidates were assessed from their public dataset cards and descriptions. **Only the selected dataset was downloaded and inspected row by row** (Step 1a below); statements about the other candidates should be re-checked before you rely on them.

| Candidate | R1 size | R2 pairs | R3 labels | R4 IT fit | R5 licence | R6 access | R7/R8 | Verdict |
|---|---|---|---|---|---|---|---|---|
| **`Tobi-Bueck/customer-support-tickets`** | ✅ ~61.8k rows | ✅ `body` + `answer` | ✅ `queue`, `type`, `priority` | ✅ partial: IT queues exist, mixed with other departments | ✅ CC BY-NC 4.0, **non-commercial** | ✅ Hub | ✅ synthetic, PII scan below | **Selected** |
| `Console-AI/IT-helpdesk-synthetic-tickets` | ~ synthetic IT tickets | ❌ a demo built on it states the data has no solution text *(unverified)* | ~ | ✅ pure IT | ? licence not checked | ✅ Hub | ✅ synthetic | Backup: seed tickets for **teacher-generated** resolutions |
| `bitext/Bitext-customer-support-llm-chatbot-training-dataset` | ✅ | ✅ | ~ intents, not routing queues | ❌ retail support (orders, refunds) | ✅ Apache 2.0 | ✅ Hub | ✅ | Not IT; useful only as a tone / refusal reference |
| `Harshdj18/help-desk-tickets` | ❌ ~3k-row free sample, full set is paid | ❌ structured ITSM fields (e.g. resolution times), no reply text | ✅ | ✅ IT | ? | ❌ paywall for the full set | ✅ | Skip |
| Zia incident-response datasets | ✅ very large | ❌ security-incident logs, not tickets with replies | ✅ | ~ security, not helpdesk | ❌ proprietary, gated (contact details required) | ❌ | ✅ | Skip: wrong task, licence, access |
| Real internal tickets (company data) | ? | ✅ | ✅ | ✅ | ❌ needs de-identification and approval | ❌ | ❌ | Not usable for a shared course project |

(✅ meets the requirement, ~ partly or unknown, ❌ does not meet it. This is an assessment, not a measurement.)

### 3. Decision and reasoning

**`Tobi-Bueck/customer-support-tickets`** is the only candidate found that satisfies R1, R2 and R3 **together**: large enough to survive heavy filtering, with an agent reply for every ticket, and with three label columns that make objective evaluation possible. Its weaknesses (R4 only partly, non-commercial licence, synthetic text) are real but manageable and are written into the limitations below.

**Why not generate our own data with a teacher model?** It is a valid option (the brief allows it), but it needs a strong teacher, a validation step, and it adds a source of bias that is hard to separate from the experiment (the teacher's style would become "the truth"). Starting from an existing labelled set gives a cleaner baseline; teacher generation stays available as a later augmentation (use the Console-AI tickets as seeds).

### 4. What the data actually looks like (evidence from Step 1a and Step 1b in this notebook)

| Fact | Value | What we do about it |
|---|---|---|
| Raw size | **61,765 rows × 16 columns** | Plenty of headroom for filtering |
| Languages | German 33,504 (54%), English 28,261 (46%) | Keep English only (R7) |
| Missing `type` | 13,178 rows (21%) | Rows without `type`, `priority`, `queue`, `body` or `answer` are dropped, because they cannot be scored |
| `priority` values in the raw data | medium 23,378, high 21,925, low 12,765, critical 1,914, very_low 1,783 | After filtering only `high`, `medium`, `low` remain (the notebook prints 3 priorities) |
| Queues in the raw data | 50+ queues, from `Technical Support` (14,186) and `Product Support` (8,960) down to many small topic queues | Only IT-style queues are kept (Step 1b) |
| IT-style queues after all filters | **Technical Support 6,616 · IT Support 2,731 · Service Outages and Maintenance 902 = 10,249 tickets** | Enough for 1,500 train + 100 val + 180 test with plenty of spare tickets (which Week 3 uses as its knowledge base) |
| Other IT-looking queues | e.g. `IT & Technology/Security Operations`, `…/Network Infrastructure`, `…/Hardware Support` appear in the raw counts (about 260–365 rows each) | They did not keep ≥ 60 tickets after the English / complete-row / length filters, so the automatic rule drops them. Use `QUEUE_OVERRIDE` if you want to try them |
| Text length (after filters) | median ticket 376 characters, median reply 374 characters | Short texts: fast to train on, and a whole ticket fits easily in a prompt |
| Template tokens in replies | 60.1% of replies contain tokens like `[Name]` | A known artefact of synthetic data; error analysis watches for the model copying them |
| Personal data | 1 ticket body and 64 replies contained an email or phone number | Masked with `<EMAIL>` / `<PHONE>` before anything is saved (Step 1c) |
| Near-duplicate leakage | 12 training-pool tickets removed for being ≥ 90% similar to a test ticket | Prevents the model from being credited for memorising (Step 2b) |

### 5. Known limitations (copy these into the model card)

- **Synthetic, non-commercial data.** Fine for coursework; **not** licensed for a commercial deployment. The model card must say so.
- **Labels and replies are LLM-generated.** Accuracy has a ceiling below 100%, and replies are often templated, so ROUGE-L will look low even for good answers.
- **The label boundary is a dataset convention.** `IT Support` vs `Technical Support` overlap semantically; which one a ticket gets depends on how the dataset was generated, not on an obvious rule. The Week 1 results below show exactly this problem.
- **Mixed departments.** Even the IT queues come from a general customer-support dataset, so some tickets are not typical enterprise IT.
- **Queue names vary between dataset versions**, which is why queue selection is data-driven, printed, and frozen in `manifest.json`.

### 6. If you want to change the dataset later

1. Change `DATASET_ID` and the column mapping in Step 1a, and re-check the required-columns assertion.
2. Use `QUEUE_OVERRIDE` to force a specific set of queues.
3. The freeze guard will (correctly) refuse to overwrite the frozen split; you must set `FORCE_REFREEZE = True`, and then **re-run Weeks 1, 2 and 3**, because every number depends on the frozen data.

### Step 1a · Load the raw dataset and take a first look

**What the next cell does**

Before cleaning or modelling anything, a good data scientist just *looks at the data*. This cell downloads the support-ticket dataset from the Hugging Face Hub, converts it to a pandas table, and prints a quick overview.

**Step by step**

1. `load_dataset(...)` fetches the `train` split (the dataset only ships one split) and caches it locally, so the next run is instant. `.to_pandas()` turns it into a normal DataFrame we can slice and filter.
2. **Schema check.** We list the 7 columns the rest of the notebook depends on and `assert` that they all exist. If the dataset author renames a column in a future version, we get a clear error *here* instead of a confusing crash 10 cells later.
3. We print the table shape, then count the values of `language`, `type` and `priority`, and list every `queue` with its ticket count. This is how we learn which categories exist and how balanced they are.
4. `raw.sample(3, random_state=SEED)` shows three random tickets. The fixed seed means you see the same three every run.

**What to look for in the output:** how many rows there are, which languages are present, whether any `priority` or `type` is very rare, and which queue names look IT-related. The next cell uses exactly that information to decide what to keep.

In [3]:
from IPython.display import display
from datasets import load_dataset

# Download (or read from the local cache) the dataset and convert it to a pandas DataFrame
raw = load_dataset(DATASET_ID, split="train").to_pandas()
# Fail fast if the dataset no longer has the columns the rest of the notebook relies on
REQUIRED = ["subject", "body", "answer", "type", "queue", "priority", "language"]
missing = set(REQUIRED) - set(raw.columns)
assert not missing, f"Dataset schema changed, missing: {missing}"

print("shape:", raw.shape)
# dropna=False also counts missing values, so hidden gaps in the labels show up in the output
for c in ["language", "type", "priority"]:
    print(raw[c].value_counts(dropna=False).head(10).to_string(), "\n")
print("queues:\n", raw["queue"].value_counts().to_string())
# A tiny random peek at real rows; fixed seed so it is the same sample every run
display(raw.sample(3, random_state=SEED)[REQUIRED])

shape: (61765, 16)
language
de    33504
en    28261 

type
Incident    19444
Request     13950
NaN         13178
Problem     10196
Change       4997 

priority
medium      23378
high        21925
low         12765
critical     1914
very_low     1783 

queues:
 queue
Technical Support                         14186
Product Support                            8960
Customer Service                           7420
IT Support                                 5725
Billing and Payments                       4874
Returns and Exchanges                      2438
Service Outages and Maintenance            1912
Sales and Pre-Sales                        1490
Human Resources                             914
General Inquiry                             668
Pets & Animals/Pet Services                 386
News                                        383
IT & Technology/Security Operations         365
Autos & Vehicles/Sales                      364
Health/Medical Services                     362
Home & Garden

,subject,body,answer,type,queue,priority,language
59903,Sicherheitsverstoß im Krankenhaus-System,"Es wurde ein Sicherheitsverstoß festgestellt, ...","Wir werden dringend Maßnahmen ergreifen, um di...",Incident,Technical Support,high,de
24200,Trouble with WhatsApp Desktop,Experienced technical difficulties during a ma...,"Hello [Name],\n\nWe hope this email finds you ...",Incident,Technical Support,medium,en
35048,Sporadische Scannerfehler am Point of Sale,"Sehr geehrtes Support-Team,\n\nich schreibe Ih...",NaN,NaN,Shopping/Retail Stores,low,de


### Step 1b · Clean the data and choose the IT queues

**What the next cell does**

The raw dataset contains ~60k tickets in several languages and many departments. Our assistant is an *IT helpdesk* assistant that answers in English, so this cell narrows the data down to the tickets we actually want, and removes rows that would hurt training or evaluation.

**Step by step**

1. **Language filter.** Keep only rows whose `language` equals `LANG` (`"en"`).
2. **Drop incomplete rows.** A ticket without a body, answer, queue, type or priority cannot be used as a training example or scored, so it goes.
3. **Tidy the text.** Fill empty values, convert everything to strings, trim stray whitespace, and lower-case `priority` so `High` and `high` are not treated as two different labels.
4. **Length filter.** Remove tickets or replies that are extremely short (nothing to learn from) or extremely long (slow and expensive to train on).
5. **Exact-duplicate removal.** We normalise each body (lower-case, collapse whitespace) and drop repeats. This matters because a ticket that appears twice could land in both train and test, which would inflate the scores.
6. **Queue selection, driven by the data.** Unless you set `QUEUE_OVERRIDE`, we keep queues whose names match `QUEUE_REGEX` (words like *network*, *security*, *software*). If fewer than 3 match, we fall back to the biggest queues. Queues with too few tickets are dropped. The chosen queues are **printed** so the decision can be audited rather than hidden in code.
7. **Give every row a stable ID.** `uid` (`t00000`, `t00001`, ...) lets us track a ticket across splits, predictions and error analysis.
8. **Size sanity check.** If there are not enough rows for the planned train/val/test sizes, `N_TRAIN` is reduced automatically and a warning is printed.

**What to look for in the output:** which queues were selected, how many tickets each has, and the final row count.

In [4]:
# Step 1: keep only the target language. astype(str).str.lower() makes the match tolerant of 'EN' vs 'en'
df = raw[raw["language"].astype(str).str.lower() == LANG].copy()
# Step 2: a row missing any key field is unusable as a training example, so drop it
df = df.dropna(subset=["body", "answer", "queue", "type", "priority"])
# Step 3: normalise text columns (no NaNs, no leading/trailing spaces) so equal labels really compare as equal
for c in ["subject", "body", "answer", "queue", "type", "priority"]:
    df[c] = df[c].fillna("").astype(str).str.strip()
df["priority"] = df["priority"].str.lower()

# Step 4: length filter on both the ticket body and the agent reply (between() is inclusive on both ends)
df = df[df["body"].str.len().between(MIN_BODY_CHARS, MAX_BODY_CHARS)
        & df["answer"].str.len().between(MIN_ANS_CHARS, MAX_ANS_CHARS)]

# exact-duplicate tickets (normalised body) would leak across splits
# the temporary _key is the body in lower case with runs of whitespace collapsed to one space
df["_key"] = df["body"].str.lower().str.replace(r"\s+", " ", regex=True)
df = df.drop_duplicates("_key").drop(columns="_key")

# queue selection: IT-style names, else top-K by volume; print so the choice is auditable
qc = df["queue"].value_counts()          # tickets per queue, largest first
if QUEUE_OVERRIDE:
    IT_QUEUES = list(QUEUE_OVERRIDE)       # manual choice wins over the automatic one
else:
    IT_QUEUES = [q for q in qc.index if re.search(QUEUE_REGEX, q)]
    if len(IT_QUEUES) < MIN_QUEUES:
        IT_QUEUES = list(qc.index[:MAX_QUEUES])   # not enough name matches -> fall back to the biggest queues
    # drop tiny queues, then cap the total so the label space stays manageable
    IT_QUEUES = [q for q in IT_QUEUES if qc[q] >= MIN_PER_QUEUE][:MAX_QUEUES]
assert len(IT_QUEUES) >= 2, "Too few usable queues; set QUEUE_OVERRIDE"

# keep only the chosen queues and give each ticket a stable ID (t00000, t00001, ...) used in every later step
df = df[df["queue"].isin(IT_QUEUES)].reset_index(drop=True)
df["uid"] = [f"t{i:05d}" for i in range(len(df))]
need = N_TEST + N_VAL + N_TRAIN
if len(df) < need:
    # small dataset: shrink the training set but still leave a 100-row cushion for dedup losses
    N_TRAIN = max(0, len(df) - N_TEST - N_VAL - 100)
    print(f"WARNING: only {len(df)} rows; N_TRAIN reduced to {N_TRAIN}")

print("selected queues:", IT_QUEUES)
print(df["queue"].value_counts().to_string())
print("rows after filtering:", len(df))

selected queues: ['Technical Support', 'IT Support', 'Service Outages and Maintenance']
queue
Technical Support                  6616
IT Support                         2731
Service Outages and Maintenance     902
rows after filtering: 10249


### Step 1c · Privacy scan, redaction and a dataset "health check"

**What the next cell does**

Support tickets often contain personal data such as email addresses and phone numbers. We do not want a model to memorise those, and we do not want them in files that get shared with the team. So this cell (1) finds and masks such data, and (2) produces a compact report that shows whether the dataset is suitable for the project.

**Step by step**

1. **Three regular expressions.** `EMAIL` matches things like `name@company.com`. `PHONE` matches long digit sequences with optional `+`, spaces, brackets and dashes. `PLACEHOLDER` matches template tokens such as `<name>` or `[Customer]` that are sometimes left in the reply text.
2. `redact()` replaces emails with `<EMAIL>` first, then phone numbers with `<PHONE>`.
3. **Count before cleaning.** `pii_before` records how many body and answer cells contained an email or phone *before* we mask them. That number is evidence for the report that the scan really found something.
4. **Apply the redaction** to `subject`, `body` and `answer`.
5. **Suitability table (`suit`).** Row count, number of distinct queues, types and priorities, the smallest and largest queue share (a quick balance check), median ticket and reply length, and the fraction of replies that still contain template placeholders.
6. **Cross-tab of queue x priority.** A grid showing how many tickets of each priority sit in each queue. Empty or very lopsided cells warn us that some label combinations will be hard to learn.

**What to look for in the output:** a queue share that is not extremely skewed, sensible median lengths, and the PII hit counts. A high placeholder share tells you the replies are templated, which limits how well ROUGE can reflect real quality.

In [5]:
# Patterns for personal data. PHONE needs 10+ digits/separators so short numbers like ticket IDs aren't masked.
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"(?<!\w)\+?\d[\d\s().-]{8,}\d(?!\w)")
# Template tokens such as <name> or [Customer] that some replies still contain (we only measure these)
PLACEHOLDER = re.compile(r"<[^<>\n]{1,30}>|\[[A-Za-z _]{2,30}\]")

def redact(s):
    # emails first, then phones, so digits inside an email can't be mistaken for a phone number
    return PHONE.sub("<PHONE>", EMAIL.sub("<EMAIL>", s))

# Count how many cells held PII BEFORE masking: this is the evidence that the scan found something
pii_before = {c: int(df[c].str.contains(EMAIL).sum() + df[c].str.contains(PHONE).sum()) for c in ["body", "answer"]}
for c in ["subject", "body", "answer"]:
    df[c] = df[c].map(redact)

# Dataset health check: normalize=True turns counts into fractions, so min/max show how lopsided the queues are
share = df["queue"].value_counts(normalize=True)
suit = pd.Series({
    "rows": len(df),
    "queues": df["queue"].nunique(), "types": df["type"].nunique(), "priorities": df["priority"].nunique(),
    "queue_share_min": round(share.min(), 3), "queue_share_max": round(share.max(), 3),
    "body_chars_median": int(df["body"].str.len().median()),
    "answer_chars_median": int(df["answer"].str.len().median()),
    "answers_with_placeholders": round(df["answer"].str.contains(PLACEHOLDER).mean(), 3),
    "pii_hits_before_redaction(body,answer)": str(tuple(pii_before.values())),
})
print(suit.to_string())
# Queue x priority grid: reveals empty or very lopsided label combinations
print("\nlabel cross-tab (queue x priority):")
print(pd.crosstab(df["queue"], df["priority"]).to_string())

rows                                        10249
queues                                          3
types                                           4
priorities                                      3
queue_share_min                             0.088
queue_share_max                             0.646
body_chars_median                             376
answer_chars_median                           374
answers_with_placeholders                   0.601
pii_hits_before_redaction(body,answer)    (1, 64)

label cross-tab (queue x priority):
priority                         high  low  medium
queue                                             
IT Support                       1287  260    1184
Service Outages and Maintenance   638  116     148
Technical Support                3840  762    2014


## 2 · Target schema and prompt format

The model must return **one JSON object**:

```json
{"queue": "...", "type": "...", "priority": "...", "resolution": "..."}
```

- `queue`, `type`, `priority` come straight from the dataset labels, so they give cheap exact-match and macro-F1 scores.
- `resolution` is the agent `answer`, scored with ROUGE-L here and with an LLM judge later.
- The same system prompt and chat-template format is reused for SFT in Week 2, so baseline and fine-tuned runs differ only in the weights.

### Step 2a · Build the system prompt and the helper functions

**What the next cell does**

A language model only does what its instructions say, so this cell writes those instructions and defines three small helpers that turn a table row into model input and model output. The same helpers are reused for the baseline now and for fine-tuning in Week 2, which is what keeps the comparison honest.

**The pieces**

- **Label vocabularies.** `QUEUES`, `TYPES` and `PRIORITIES` are read *from the data* (for example `priority` -> `high`, `low`, `medium`). Because the lists come from the dataset, the prompt can never drift away from the real labels.
- **`SYSTEM` prompt.** An f-string that tells the model: you are an IT helpdesk assistant, answer with exactly one JSON object, use these four keys, and pick the queue/type/priority only from the allowed values. `json.dumps(...)` inserts the allowed values as neat JSON lists. It also sets two safety rules: never ask for passwords and never invent policy details.
- **`ticket_text(r)`** formats a row as the *user message*: `Subject: ...`, a blank line, then the body. If a ticket has no subject it prints `(none)`.
- **`target_json(r)`** builds the *ideal answer* for a ticket: a JSON string with `queue`, `type`, `priority`, and `resolution` (the real agent reply from the `answer` column).
- **`sft_messages(r)`** puts it all together as a chat conversation: system message, user message (the ticket) and assistant message (the ideal answer). This is the standard "messages" format that chat models and fine-tuning libraries understand.

**What to look for in the output:** the printed system prompt (check the allowed label lists look right) and the first training example in chat format.

In [6]:
# Label vocabularies are read from the data (not typed by hand) so prompt and dataset can never disagree
QUEUES = list(IT_QUEUES)
TYPES = sorted(df["type"].unique())
PRIORITIES = sorted(df["priority"].unique())
KEYS = ["queue", "type", "priority", "resolution"]   # the exact keys the model must output, in this order

# The system prompt: the model's "job description". json.dumps() injects the allowed label lists as JSON.
SYSTEM = f"""You are an enterprise IT helpdesk assistant. Read the support ticket and answer with exactly one JSON object and no other text.
Keys:
- "queue": one of {json.dumps(QUEUES)}
- "type": one of {json.dumps(TYPES)}
- "priority": one of {json.dumps(PRIORITIES)}
- "resolution": a concise, professional reply to the user with concrete next steps (under 150 words). Never ask for passwords and never invent policy details."""

def ticket_text(r):
    # What the model reads as the user message: subject line, blank line, body
    return f"Subject: {r['subject'] or '(none)'}\n\n{r['body']}"

def target_json(r):
    # The ideal answer for a ticket. The dataset column is called `answer`, but our schema key is `resolution`
    return json.dumps({k: r[k] if k != "resolution" else r["answer"] for k in KEYS}, ensure_ascii=False)

def sft_messages(r):
    # One full training example in chat format: system -> user (ticket) -> assistant (ideal JSON answer)
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": ticket_text(r)},
            {"role": "assistant", "content": target_json(r)}]

# Sanity check: print the prompt and the first example (cut at 1200 chars) to eyeball the format
print(SYSTEM, "\n")
print(json.dumps(sft_messages(df.iloc[0].to_dict()), indent=2, ensure_ascii=False)[:1200])

You are an enterprise IT helpdesk assistant. Read the support ticket and answer with exactly one JSON object and no other text.
Keys:
- "queue": one of ["Technical Support", "IT Support", "Service Outages and Maintenance"]
- "type": one of ["Change", "Incident", "Problem", "Request"]
- "priority": one of ["high", "low", "medium"]
- "resolution": a concise, professional reply to the user with concrete next steps (under 150 words). Never ask for passwords and never invent policy details. 

[
  {
    "role": "system",
    "content": "You are an enterprise IT helpdesk assistant. Read the support ticket and answer with exactly one JSON object and no other text.\nKeys:\n- \"queue\": one of [\"Technical Support\", \"IT Support\", \"Service Outages and Maintenance\"]\n- \"type\": one of [\"Change\", \"Incident\", \"Problem\", \"Request\"]\n- \"priority\": one of [\"high\", \"low\", \"medium\"]\n- \"resolution\": a concise, professional reply to the user with concrete next steps (under 150 word

### Step 2b · Split the data, guard against leakage, and freeze it

**What the next cell does**

This is the most important cell of Week 1. It divides the tickets into **train**, **validation** and **test** sets and then *freezes* them to disk with checksums, so every later experiment is guaranteed to use exactly the same data. Think of the test set as a sealed final exam: nobody may look at it while studying.

**Step by step**

1. **Carve out the test set first**, using a *stratified* split on `queue`. Stratified means each queue keeps the same proportion in the test set as in the full data, so small queues are not accidentally left out.
2. **Leakage guard.** Even after removing exact duplicates, templated tickets can be *almost* identical (one changed word). If such a near-copy is in training while its twin is in the test set, the model gets credit for memorising rather than understanding. We turn each ticket body into a TF-IDF vector (word and two-word patterns), measure the cosine similarity between every pool ticket and every test ticket, and drop any pool ticket whose best match is >= `DEDUP_SIM` (0.90).
3. **Cap the pool and split off validation.** If more rows remain than we need, we sample down to `N_TRAIN + N_VAL`, again stratified. The rest is divided into train and validation. Validation is for tuning choices during fine-tuning; test is only for the final score.
4. **Hard safety checks (`assert`).** No ticket ID and no ticket body may appear in both train/val and test. If this ever fails, the notebook stops instead of silently giving inflated scores.
5. **Build the files in memory.** `to_jsonl()` turns each split into JSONL text (one JSON record per line, including the chat `messages`), and we compute each text's **SHA-256 hash**, a fingerprint that changes if even one character differs.
6. **`manifest.json`** records the dataset, licence, seed, label lists, split sizes, filters and the three hashes. Committing this file is the Week 1 gate: anyone can later re-hash the files and prove the data was not touched.
7. **Freeze guard.** If a manifest already exists, the new hashes must match it. If they do not (for example you changed `N_TRAIN` for an experiment), the cell raises an error **before writing anything**, so the frozen files can never be silently replaced. Running the cell again with the original settings is always safe: it rewrites identical files. `FORCE_REFREEZE = True` bypasses the guard for a deliberate new freeze.
8. Only then do we write the three JSONL files and the manifest, and print the hashes and the queue proportions in train vs test; the two columns should be almost identical.

**What to look for in the output:** how many near-duplicates were dropped, the final train/val/test sizes, and matching queue proportions.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# test set is carved out FIRST so nothing downstream can touch it
# stratify=queue keeps the same queue mix in the test set as in the full data
pool, test = train_test_split(df, test_size=N_TEST, stratify=df["queue"], random_state=SEED)

# near-duplicate leakage guard: drop pool rows too similar to any test ticket
# TF-IDF turns each ticket into a weighted word/phrase vector; fitting on pool+test shares one vocabulary
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(pd.concat([pool["body"], test["body"]]))
# (pool x test) matrix product of normalised vectors = cosine similarity; .max(axis=1) keeps each pool row's closest test ticket
sim = (vec.transform(pool["body"]) @ vec.transform(test["body"]).T).max(axis=1).toarray().ravel()
leaky = sim >= DEDUP_SIM
print(f"dropped {int(leaky.sum())} near-duplicate pool rows (cosine >= {DEDUP_SIM})")
pool = pool[~leaky]

# sample the pool down to the size we need (the `_` throwaway holds the unused remainder), then split train/val
if len(pool) > N_TRAIN + N_VAL:
    pool, _ = train_test_split(pool, train_size=N_TRAIN + N_VAL, stratify=pool["queue"], random_state=SEED)
train, val = train_test_split(pool, test_size=N_VAL, stratify=pool["queue"], random_state=SEED)
train, val, test = [d.reset_index(drop=True) for d in (train, val, test)]

# Hard guarantees: no shared ticket IDs or identical bodies between train/val and test, or we stop right here
assert set(train["uid"]).isdisjoint(test["uid"]) and set(val["uid"]).isdisjoint(test["uid"])
assert set(train["body"]).isdisjoint(set(test["body"]))
print({"train": len(train), "val": len(val), "test": len(test)})

# freeze: JSONL + SHA-256 manifest (the Week 1 gate artefact)
def to_jsonl(d):
    # One JSON record per line: the row plus its ready-made chat `messages`
    return "".join(json.dumps({**r, "messages": sft_messages(r)}, ensure_ascii=False) + "\n" for r in d.to_dict("records"))

# Build the exact file contents in memory FIRST, so we can compare with any earlier freeze before touching a file.
# The SHA-256 "fingerprint" changes if even one character differs.
payload = {n: to_jsonl(d) for n, d in [("train", train), ("val", val), ("test", test)]}
sha256 = {n: hashlib.sha256(t.encode("utf-8")).hexdigest() for n, t in payload.items()}

# The manifest is the "receipt" for the frozen data: what we used, how we filtered it, and each file's hash
manifest = {
    "dataset": DATASET_ID, "license": "CC BY-NC 4.0", "language": LANG, "seed": SEED,
    "queues": QUEUES, "types": TYPES, "priorities": PRIORITIES,
    "counts": {"train": len(train), "val": len(val), "test": len(test)},
    "sha256": sha256,
    "filters": {"body_chars": [MIN_BODY_CHARS, MAX_BODY_CHARS], "answer_chars": [MIN_ANS_CHARS, MAX_ANS_CHARS], "dedup_sim": DEDUP_SIM},
}

# Freeze guard: if a manifest already exists, the new data must reproduce it exactly, otherwise stop BEFORE writing.
# (Re-running this cell with a different N_TRAIN, seed or filter would otherwise silently replace the frozen files.)
mpath = OUT / "manifest.json"
if mpath.exists() and not FORCE_REFREEZE:
    frozen = json.loads(mpath.read_text())["sha256"]
    changed = [n for n in sha256 if sha256[n] != frozen.get(n)]
    if changed:
        raise RuntimeError(f"Refusing to overwrite the frozen split(s) {changed}: the data differs from manifest.json. "
                           "Restore the settings you froze with, or set FORCE_REFREEZE = True for a deliberate new freeze "
                           "(Week 2 and every baseline number would then have to be re-done).")
for n, text in payload.items():
    (OUT / f"{n}.jsonl").write_text(text, encoding="utf-8")
mpath.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest["sha256"], indent=2))
# Side-by-side queue proportions: train and test should look almost identical if the stratified split worked
print(pd.concat({"train": train["queue"].value_counts(normalize=True),
                 "test": test["queue"].value_counts(normalize=True)}, axis=1).round(3).to_string())

dropped 12 near-duplicate pool rows (cosine >= 0.9)
{'train': 1500, 'val': 100, 'test': 180}
{
  "train": "1dee2cc5259a55b0ae7d5060ce344e1e61cc64e5c9883b88980f5fc696d6edff",
  "val": "9e88d592445e38a88652c40b4a359a758c1e1f85ffb52ed2f6633bd1f2d43b27",
  "test": "4ed79e20ab0be0de83579cc73ea406f055df6aab304d1d3dabfb76f6ccdfd837"
}
                                 train   test
queue                                        
Technical Support                0.645  0.644
IT Support                       0.267  0.267
Service Outages and Maintenance  0.088  0.089


## 3 · Baseline design

Two prompting baselines on the **same frozen test set** with greedy decoding:

- **Zero-shot**: system prompt (schema + allowed values) and the ticket.
- **Few-shot**: the same, plus $k$ worked examples drawn from the **train** split only (one per most-frequent queue, median-length reply), so the test set stays untouched.

These are the bars the fine-tuned model (Week 2) and fine-tuned + RAG (Week 3) must beat. Keep decoding, precision and prompts identical when you re-run the evaluation later.

### Step 3a · The scoring toolkit: JSON extractor, row scorer and summary

**What the next cell does**

It defines (but does not yet run) the three functions that turn raw model text into numbers. Writing the scoring logic once and reusing it is what lets us compare zero-shot, few-shot and (later) the fine-tuned model on exactly the same ruler.

**The three functions**

1. **`extract_json(text)`** finds the first well-formed `{...}` object in the model's output. Models do not always obey "JSON only": they may wrap the answer in a ```json fence or add a polite sentence first. The function walks through the text character by character, tracks how deeply nested the braces are, and ignores braces inside quoted strings (that is what the `in_str` and `esc` flags are for). When the braces balance, it tries `json.loads`; if that fails it moves on to the next `{`. It returns `None` when nothing parseable is found.
2. **`score_row(...)`** grades one prediction against the gold ticket. It records:
   - `parsed`: did we find JSON at all?
   - `schema_valid`: does it have exactly the 4 required keys, with queue/type/priority from the allowed lists and a non-empty `resolution`?
   - `queue_ok`, `type_ok`, `priority_ok`: are the labels exactly right?
   - `rougeL`: overlap between the generated `resolution` and the reference reply (0 if there is no usable reply). ROUGE-L is based on the longest common word sequence; stemming means *resetting* and *reset* count as similar.
3. **`summarize(...)`** averages those per-row results into the headline numbers: parse rate, schema-valid rate, accuracy per field, exact match of all three labels, **macro-F1** for queue and priority, and mean ROUGE-L.

**Why macro-F1, not just accuracy?** Our queues are imbalanced (one queue has most tickets). A lazy model that always answers the biggest queue would still score high accuracy. Macro-F1 computes F1 *per class* and averages them equally, so small classes count just as much, which makes it a much more honest metric here. `zero_division=0` simply scores a class as 0 when the model never predicts it.

Nothing prints from this cell; it only defines functions for the cells below.

In [8]:
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score
# ROUGE-L = longest common word sequence between reply and reference; use_stemmer treats word variants as equal
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def extract_json(text):
    """First balanced {...} object in text that parses as JSON (tolerates fences / chatter)."""
    # Scan from each "{" and count nesting depth; braces inside quoted strings must not be counted
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False   # esc = previous char was a backslash (so the next quote is escaped)
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                esc = (not esc) and ch == "\\"
                if ch == '"' and not esc:
                    in_str = False
            elif ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    # braces balanced: try to parse this slice; if it isn't valid JSON, try the next "{"
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None

def score_row(pred_text, gold, queues, types, priorities):
    obj = extract_json(pred_text)
    parsed = isinstance(obj, dict)           # JSON arrays or bare values don't count as a valid answer
    g = obj if parsed else {}
    # schema_valid is strict: exactly the 4 keys, allowed label values, and a non-empty text reply
    valid = (parsed and set(g) == {"queue", "type", "priority", "resolution"}
             and g["queue"] in queues and g["type"] in types and g["priority"] in priorities
             and isinstance(g["resolution"], str) and g["resolution"].strip() != "")
    res = g.get("resolution") if isinstance(g.get("resolution"), str) else ""
    return {
        "parsed": parsed, "schema_valid": bool(valid),
        # "INVALID" marks a missing label so it counts as wrong (and shows up in the confusion table later)
        "pred_queue": g.get("queue", "INVALID"), "pred_type": g.get("type", "INVALID"),
        "pred_priority": g.get("priority", "INVALID"),
        "queue_ok": g.get("queue") == gold["queue"], "type_ok": g.get("type") == gold["type"],
        "priority_ok": g.get("priority") == gold["priority"],
        # no usable reply -> ROUGE-L is 0, so broken outputs are penalised instead of ignored
        "rougeL": _rouge.score(gold["answer"], res)["rougeL"].fmeasure if res else 0.0,
    }

def summarize(scored, gold_df, queues, priorities):
    s = scored
    # .mean() of a True/False column = the fraction of rows that are True, i.e. a rate
    return {
        "n": len(s),
        "json_parse_rate": s["parsed"].mean(), "schema_valid_rate": s["schema_valid"].mean(),
        "queue_acc": s["queue_ok"].mean(), "type_acc": s["type_ok"].mean(), "priority_acc": s["priority_ok"].mean(),
        "labels_exact_match": (s["queue_ok"] & s["type_ok"] & s["priority_ok"]).mean(),   # all 3 labels right at once
        # macro-F1 averages F1 over classes equally, so rare queues matter as much as the big one
        "queue_macro_f1": f1_score(gold_df["queue"], s["pred_queue"], labels=queues, average="macro", zero_division=0),
        "priority_macro_f1": f1_score(gold_df["priority"], s["pred_priority"], labels=priorities, average="macro", zero_division=0),
        "rougeL_resolution": s["rougeL"].mean(),
    }

### Step 4 · Before touching the GPU: memory helpers and a pre-flight check

**What the next cell does**

Loading a 7B model takes a minute or two and then fails with `CUDA out of memory` if another program (for example a notebook kernel you forgot about) is already sitting on the GPU. This cell defines three small helpers and uses one of them right away to check **before** the slow load. The same helpers are used by the cleanup cell at the end, and an identical copy lives in the Week 2 notebook so that each notebook is self-contained.

**The helpers**

- `gpu_report(tag)` prints, for every GPU, how much memory is in use **in total** (the driver's view, all programs together) next to how much belongs to **this kernel's own** tensors. If the total is high but "this kernel" is near zero, another process is the culprit.
- `free_gpu(tag)` hands memory back. It does four things in order: clears the saved traceback of an earlier error (a traceback keeps every variable of the failed cell alive, including the model!), runs Python's garbage collector, calls `torch.cuda.empty_cache()` to return PyTorch's cached memory to the driver, and `ipc_collect()` to clean up shared-memory leftovers. **Delete the big objects (`del model`) first**: memory can only be released when nothing refers to it.
- `check_gpu_free(min_free_gib)` is the pre-flight check. It **refuses to start** if another process holds more than 2 GiB on any GPU (two jobs cannot share a 15 GB T4; the second would crash part-way through), and also raises if less than `min_free_gib` is free in total. The error message says what to do. `allow_shared=True` overrides the first check.

- `release_model(model)` swaps every weight of a model for an empty tensor. A plain `del model` was **not enough** in our Week 2 tests: after QLoRA training a 2 GB float32 copy of the embedding matrix stayed on the GPU because a library still referenced the module. This frees the GPU storage regardless; the cleanup cell uses it.

**What to look for in the output:** one line per GPU. On a clean machine you should see roughly 0 GiB used. If it stops with an error, run `nvidia-smi` in a terminal, find the other Python process, and shut down that notebook's kernel.

In [9]:
import gc, sys, torch

def gpu_report(tag=""):
    # mem_get_info = driver's view (all processes); memory_allocated/reserved = only this kernel's PyTorch memory
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"{tag}GPU {i}: {(total - free) / 2**30:.1f}/{total / 2**30:.1f} GiB used | "
              f"this kernel: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated, {torch.cuda.memory_reserved(i) / 2**30:.1f} GiB reserved")

def free_gpu(tag=""):
    """Release cached GPU memory. `del` the big objects (model, trainer, ...) BEFORE calling this."""
    sys.last_type = sys.last_value = sys.last_traceback = None   # an old error's traceback keeps the model alive
    gc.collect()                                                 # drop objects nothing refers to any more
    torch.cuda.empty_cache(); torch.cuda.ipc_collect()           # return cached memory to the driver
    gpu_report(tag)

def release_model(m):
    """Free the GPU storage of every weight in `m`, even if some library still references the module object.
    (Measured in Week 2: after QLoRA training a 2 GB float32 copy of the embedding matrix stayed on the GPU after `del model`.)"""
    with torch.no_grad():
        for t in list(m.parameters()) + list(m.buffers()):
            if hasattr(t, "quant_state"): t.quant_state = None   # 4-bit weights keep their scale tensors here
            t.data = torch.empty(0, dtype=t.dtype, device="cpu")  # swap the big GPU tensor for an empty CPU one

def check_gpu_free(min_free_gib, allow_shared=False):
    """Pre-flight: fail fast, with the fix, if the GPUs are already occupied by another process."""
    assert torch.cuda.is_available(), "GPU required: Runtime > Change runtime type > GPU"
    gpu_report("pre-flight: ")
    free_total, busy = 0.0, []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        free_total += free / 2**30
        other = (total - free) - torch.cuda.memory_reserved(i)   # used by something that is not this kernel
        if other > 2 * 2**30:
            busy.append(f"GPU {i}: ~{other / 2**30:.1f} GiB")
    if busy and not allow_shared:
        # two jobs cannot share a 15 GB T4: the second one would run out of memory part-way through
        raise RuntimeError("Another process is using the GPU(s) (" + ", ".join(busy) + "). Another notebook may still be "
                           "training, or a finished kernel still holds its model. Check `nvidia-smi`, wait for it or shut "
                           "that kernel down, then re-run. (Pass allow_shared=True to override.)")
    if free_total < min_free_gib:
        raise RuntimeError(f"Only {free_total:.1f} GiB of GPU memory is free but ~{min_free_gib} GiB is needed. "
                           "Shut down other notebook kernels that hold the GPU (find them with `nvidia-smi`), then re-run.")

check_gpu_free(min_free_gib=7)           # 4-bit 7B weights ~5.5 GB + room for activations while generating

pre-flight: GPU 0: 0.4/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved


### Step 4a · Load the base model onto the GPU (4-bit)

**What the next cell does**

This is the first cell that needs a GPU. It downloads `Qwen2.5-7B-Instruct` (about 15 GB in full precision), compresses it to 4-bit while loading, and puts it on the GPU, together with the tokenizer that converts text to numbers and back.

**Step by step**

1. **GPU check.** `torch.cuda.is_available()` must be `True`, otherwise we stop with a clear message. (On this machine this was `False` earlier because the installed PyTorch was built for a newer CUDA than the driver supports; that was fixed by installing a CUDA 12.6 build.)
2. **Optional Hugging Face login.** Gated models such as Llama need an access token. If an `HF_TOKEN` environment variable exists we log in; otherwise we skip this.
3. **Pick the number format (`dtype`).** We use `bfloat16` only on GPUs that support it **natively** (compute capability 8.0 or higher: A100, RTX 30xx/40xx, H100) and `float16` otherwise. This matters on the T4 (compute capability 7.5): it has no bfloat16 hardware, and recent PyTorch versions will still *claim* support through slow software emulation. We measured it: with emulated bfloat16 a single QLoRA training step took about 80 seconds on this machine, which would turn the full Week 2 run into many hours. `float16` runs on the T4's tensor cores. The choice only changes the precision of the calculations; the stored weights are 4-bit either way. Week 2 uses the identical rule, and the chosen dtype is saved in `baseline_results.json` so Week 2 can verify they match. `dtype` is passed to the loader as `dtype=` in **both** modes: it sets the precision of the layers that are *not* quantized (embeddings, output layer, norms). Without it, recent transformers versions use the checkpoint's own dtype (`"auto"`), so pinning it makes the precision explicit and keeps the baseline and fine-tuned comparison fair. (Older transformers versions call this argument `torch_dtype`.)
4. **4-bit quantization (`BitsAndBytesConfig`).** Storing each weight in 4 bits instead of 16 cuts memory roughly 4x, so a 7B model fits in a 15 GB T4.
   - `nf4` is a 4-bit format designed for the bell-curve shape of neural-network weights, so it loses less accuracy than plain 4-bit integers.
   - `double_quant` compresses the quantization constants as well, saving a little more memory.
   - `compute_dtype` is the precision used when the weights are temporarily expanded for the maths.
   - `device_map="auto"` lets the `accelerate` library decide where each layer goes (and spill across both GPUs if needed).
5. **Tokenizer.** `padding_side = "left"` is important for generation: when several prompts of different length are batched, the padding goes on the left so every prompt *ends* right where the model starts writing. If the tokenizer has no pad token, we reuse the end-of-text token.
6. **Load the model** and call `.eval()` to switch off training-only behaviour such as dropout.

**What to look for in the output:** a download/loading progress bar the first time, then a one-line summary showing the model, `4-bit` and the GPU name. You can watch the memory with `nvidia-smi` in a terminal. Expect roughly 5-6 GB for the weights.

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Stop early with a readable message if PyTorch can't see a GPU (loading a 7B model on CPU would take forever)
assert torch.cuda.is_available(), "GPU required: Runtime > Change runtime type > GPU"
if os.environ.get("HF_TOKEN"):                       # needed for gated models (Llama)
    from huggingface_hub import login; login(os.environ["HF_TOKEN"])

# Compute precision: bfloat16 only where the GPU has it natively (Ampere or newer). A T4 merely emulates it, which is
# painfully slow (measured ~80 s per QLoRA step), so there we use float16. Week 2 applies the identical rule.
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
# dtype is set in BOTH modes: with 4-bit loading it controls the layers that are not quantized
# (embeddings, lm_head, norms). Pinning it matches the loading recipe the QLoRA runs use, so the comparison stays fair.
kw = {"device_map": "auto", "dtype": dtype}   # device_map="auto": let accelerate place the layers on the available GPU(s)
if LOAD_4BIT:
    # NF4 4-bit weights: ~4x smaller in memory, with little accuracy loss; double-quant squeezes the constants too
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"                            # decoder-only models must be left-padded for batched generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token                    # some tokenizers ship without a pad token; reuse end-of-text
# .eval() switches off training-only behaviour (like dropout) so outputs are deterministic and consistent
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw).eval()
print(MODEL_ID, "| 4-bit" if LOAD_4BIT else f"| {dtype}", "| GPU:", torch.cuda.get_device_name(0))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen/Qwen2.5-7B-Instruct | 4-bit | GPU: NVIDIA L40S


### Step 4b · Helpers that build prompts and generate answers

**What the next cell does**

It defines two functions (no output yet): one that formats a conversation exactly the way the model expects, and one that runs a *batch* of tickets through the model and measures how long that took.

**`build_prompt(msgs)`**

Chat models are trained on a specific text layout with special markers such as `<|im_start|>system ... <|im_end|>`. Rather than hand-writing that layout, `tok.apply_chat_template` builds it from our list of messages. `add_generation_prompt=True` appends the marker that says "the assistant speaks now", so the model starts writing its answer. Some models (certain Mistral versions) reject a separate `system` message, so the `except` branch merges the system text into the first user message and tries again.

**`generate_batch(msgs_list)`**

1. `@torch.inference_mode()` turns off the bookkeeping PyTorch needs for training, saving memory and time.
2. We build one prompt per ticket and tokenize them *together* with padding (`padding=True`), so the GPU handles `GEN_BATCH` tickets in a single pass. `add_special_tokens=False` avoids adding start-of-text tokens twice, since the chat template already contains them.
3. `torch.cuda.synchronize()` is called before and after timing. GPU work is asynchronous, so without it the stopwatch would stop before the GPU has actually finished and the speed numbers would be wrong.
4. `model.generate(..., do_sample=False)` is **greedy decoding**: always pick the most likely next token. That makes results deterministic and comparable between runs. The `temperature/top_p/top_k=None` arguments just silence warnings about sampling settings we are not using.
5. The output contains the prompt followed by the new text, so we slice off the prompt (`out[:, prompt_length:]`) to keep only what the model wrote.
6. We count the generated tokens (ignoring padding) for the throughput metric, then decode token IDs back into text.

It returns four things: the generated texts, the token count per ticket, the batch time in seconds, and the padded prompt length.

In [11]:
def build_prompt(msgs):
    # Turn the message list into the model's exact chat format; add_generation_prompt=True ends with "assistant, your turn"
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:                                # templates without a system role (e.g. some Mistral versions)
        # fallback: glue the system text and the first user message into a single user message
        merged = [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}] + msgs[2:]
        return tok.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()                              # no gradients needed: faster and uses less memory
def generate_batch(msgs_list):
    # Tokenize all prompts together; padding makes them equal length so they run as one GPU batch
    enc = tok([build_prompt(m) for m in msgs_list], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    # synchronize() waits for the GPU to finish, otherwise the timer would stop before the work is really done
    torch.cuda.synchronize(); t0 = time.perf_counter()
    # do_sample=False = greedy decoding (always the most likely token) -> deterministic, repeatable results
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    # generate() returns prompt + answer; slice off the prompt part so we keep only the newly written tokens
    gen = out[:, enc["input_ids"].shape[1]:]
    n_tok = [int((g != tok.pad_token_id).sum()) for g in gen]   # real tokens per ticket (padding excluded)
    return tok.batch_decode(gen, skip_special_tokens=True), n_tok, dt, int(enc["input_ids"].shape[1])

### Step 4c · Run the zero-shot and few-shot baselines on the test set

**What the next cell does**

This is where the base model is actually put to the test. It answers every test ticket twice, once with each prompting style, and saves the raw outputs to disk. This is the longest-running cell in the notebook.

**The two baselines**

- **Zero-shot:** the model only gets the system prompt (the rules) and the ticket. It has to work out the format from the instructions alone.
- **Few-shot:** the same, but we first show it `K_SHOT` solved examples (ticket + ideal JSON answer) as earlier turns in the chat. Showing the format by example usually fixes many formatting mistakes without any training.

**The functions and the loop**

1. `pick_shots(train, k)` chooses the examples *only from the training split* (never test, which would leak answers). It takes the `k` most common queues and, for each, the ticket whose reply length is closest to the median. That gives typical, representative examples rather than odd outliers.
2. `build_msgs(row, mode)` assembles the chat: system prompt, then (for few-shot) the worked examples, then the real ticket.
3. `run_mode(mode, rows)` goes through the tickets in batches of `GEN_BATCH`, calls `generate_batch`, and records for each ticket its text, token count, batch time, batch size and prompt length. `tqdm` shows a progress bar.
4. `test_rows = test.to_dict("records")[:SMOKE_N]` takes all test tickets, or just the first `SMOKE_N` when you set a smoke-test size (`[:None]` means "everything").
5. **Resume support, with a safety check.** For each mode the results go to `preds_<mode>.jsonl`, and next to it a small `.meta.json` records a **fingerprint** of everything that influences the generated text: model, 4-bit setting, compute precision (`dtype`), token limit, batch size, number of shots, a hash of the system prompt and a hash of the frozen test set. If a file *and* a matching fingerprint exist, we load the saved answers instead of regenerating (so a disconnect does not lose finished work). If the fingerprint differs or is missing (for example the predictions were made with another precision), the cell tells you and **regenerates automatically**, so stale answers can never end up in your baseline numbers.
6. **Memory hand-back.** When both modes are done, `free_gpu()` returns the memory PyTorch cached while generating (activations and the key-value cache). The model weights stay loaded until Step 6, so you can still re-run a mode.

**What to look for in the output:** two progress bars, then one line per mode showing the file written and the number of rows (should equal the test size). While it runs, `nvidia-smi` in a terminal should show GPU utilisation going up.

In [12]:
from tqdm.auto import tqdm

# few-shot examples: train split only, one per most-frequent queue, reply length closest to the median
def pick_shots(train_df, k):
    shots = []
    for q in train_df["queue"].value_counts().index[:k]:
        c = train_df[train_df["queue"] == q]
        L = c["answer"].str.len().values
        # argmin of |length - median| = the most "typical" reply length, so the example isn't an outlier
        shots.append(c.iloc[int(np.argmin(np.abs(L - np.median(L))))].to_dict())
    return shots

SHOTS = pick_shots(train, K_SHOT)

def build_msgs(row, mode):
    msgs = [{"role": "system", "content": SYSTEM}]
    if mode == "few_shot":
        # each worked example is a fake earlier turn: the user sent a ticket, the assistant answered correctly
        for s in SHOTS:
            msgs += [{"role": "user", "content": ticket_text(s)}, {"role": "assistant", "content": target_json(s)}]
    msgs.append({"role": "user", "content": ticket_text(row)})   # the real ticket always comes last
    return msgs

def run_mode(mode, rows):
    recs = []
    # process the test tickets GEN_BATCH at a time; tqdm draws the progress bar
    for i in tqdm(range(0, len(rows), GEN_BATCH), desc=mode):
        batch = rows[i:i + GEN_BATCH]
        texts, n_tok, dt, plen = generate_batch([build_msgs(r, mode) for r in batch])
        # keep timing info per row; batch_sec is shared by the whole batch (we de-duplicate it when computing speed)
        for r, t, n in zip(batch, texts, n_tok):
            recs.append({"uid": r["uid"], "pred_text": t, "gen_tokens": n, "batch_sec": dt,
                         "batch_size": len(batch), "prompt_len": plen})
    return pd.DataFrame(recs)

# [:None] keeps every row; set SMOKE_N to a small number for a quick trial run
test_rows = test.to_dict("records")[:SMOKE_N]

# Fingerprint of everything that changes the generated text. Saved predictions are reused ONLY if it matches, so
# stale answers (older model settings, other precision, edited prompt or data) can never leak into the results.
fingerprint = {"model": MODEL_ID, "load_4bit": LOAD_4BIT, "dtype": str(dtype), "max_new_tokens": MAX_NEW_TOKENS,
               "gen_batch": GEN_BATCH, "k_shot": K_SHOT, "smoke_n": SMOKE_N,
               "system_sha": hashlib.sha256(SYSTEM.encode()).hexdigest()[:16],
               "test_sha": json.loads((OUT / "manifest.json").read_text())["sha256"]["test"][:16]}
preds = {}
for mode in ["zero_shot", "few_shot"]:
    p = OUT / f"preds_{mode}{'_smoke' if SMOKE_N else ''}.jsonl"
    meta = p.with_suffix(".meta.json")               # sits next to the predictions and records the fingerprint
    if p.exists() and meta.exists() and json.loads(meta.read_text()) == fingerprint:   # resume after a Colab disconnect
        preds[mode] = pd.read_json(p, lines=True)
    else:
        if p.exists():
            print(f"{p.name}: saved with different (or unknown) settings -> regenerating")
        preds[mode] = run_mode(mode, test_rows)
        # save right away so finished work survives a crash; the fingerprint is written last, so a half-written file is never trusted
        preds[mode].to_json(p, orient="records", lines=True, force_ascii=False)
        meta.write_text(json.dumps(fingerprint, indent=2))
    print(mode, "->", p, len(preds[mode]), "rows")

# Generation is finished: hand back the memory PyTorch cached for activations and the key-value cache.
# The model itself stays loaded (it is deleted in Step 6) in case you want to re-run a mode.
free_gpu("generation done: ")

zero_shot -> capstone_it_helpdesk/week1/preds_zero_shot.jsonl 180 rows
few_shot -> capstone_it_helpdesk/week1/preds_few_shot.jsonl 180 rows


generation done: GPU 0: 5.7/44.4 GiB used | this kernel: 5.2 GiB allocated, 5.2 GiB reserved


### Step 5a · Score the predictions and build the results table

**What the next cell does**

The previous cell produced raw text. Here we grade all of it with the toolkit from Step 3a, put the numbers for both baselines side by side in a coloured table, and save them to `baseline_results.json`. Those numbers are the **bar the fine-tuned model must beat** in Weeks 2 and 3.

**Step by step**

1. `gold` is the true test rows (same order and length as the predictions we scored).
2. For each mode (zero-shot, few-shot) we line the predictions up with the gold rows by `uid`, so a mis-ordered file can never mix up tickets. Then `score_row` grades every ticket and `summarize` averages the grades.
3. **Speed numbers.** Every row in a batch carries the *same* batch time, so simply summing `batch_sec` would count each batch several times. Dividing each row's time by its batch size and summing gives the true total wall time. From it we compute `sec_per_example` and `tokens_per_sec` (generated tokens divided by time). Also recorded: the mean output length and mean prompt length. The few-shot prompt is much longer, which is why it is slower.
4. **Results table.** One row per baseline. Percent columns are formatted as `%`, F1 and ROUGE as 3-decimal numbers, and a green gradient makes the stronger cells easy to spot.
5. **Save the evidence.** The JSON includes the settings used (model, 4-bit, compute precision `dtype`, batch size, k-shot), the **data hashes** from the manifest (proving which frozen data was scored), all results, and a timestamp. Week 2 reads the `dtype` from this file and refuses to run with a different precision, because that would make the comparison unfair.
6. If `USE_WANDB` is on, each baseline is also logged as a Weights & Biases run for experiment tracking.

**How to read the table:** `schema_valid_rate` shows whether the model follows the output format. `queue_macro_f1`/`priority_macro_f1` show classification quality. `rougeL_resolution` shows how close the replies are to the reference. Copy the key numbers into the checklist table at the bottom of the notebook.

In [13]:
from IPython.display import display
# The ground-truth rows we are grading against (just the first SMOKE_N rows when doing a smoke test)
gold = test.iloc[:len(test_rows)].reset_index(drop=True)

rows, scored_by_mode = {}, {}
for mode, p in preds.items():
    # re-order predictions to match gold by uid, so a shuffled prediction file can't pair a reply with the wrong ticket
    p = p.set_index("uid").loc[gold["uid"]].reset_index()
    sc = pd.DataFrame([score_row(t, g, QUEUES, TYPES, PRIORITIES) for t, g in zip(p["pred_text"], gold.to_dict("records"))])
    scored_by_mode[mode] = pd.concat([p, sc], axis=1)     # keep predictions + per-row scores for error analysis
    m = summarize(sc, gold, QUEUES, PRIORITIES)
    # batched HF throughput: batch wall time is shared by its rows, so count each batch once
    tot_sec = (p["batch_sec"] / p["batch_size"]).sum()
    m.update({"mean_out_tokens": p["gen_tokens"].mean(), "mean_prompt_len": p["prompt_len"].mean(),
              "sec_per_example": tot_sec / len(p), "tokens_per_sec": p["gen_tokens"].sum() / tot_sec})
    rows[mode] = m

# one row per baseline (.T flips the dict-of-dicts so modes are rows and metrics are columns)
res = pd.DataFrame(rows).T
pct = ["json_parse_rate", "schema_valid_rate", "queue_acc", "type_acc", "priority_acc", "labels_exact_match"]
# Pretty-print: rates as %, F1/ROUGE with 3 decimals, and a green gradient so higher scores stand out
display(res.style.format({**{c: "{:.1%}" for c in pct},
                          "queue_macro_f1": "{:.3f}", "priority_macro_f1": "{:.3f}", "rougeL_resolution": "{:.3f}",
                          "mean_out_tokens": "{:.0f}", "mean_prompt_len": "{:.0f}",
                          "sec_per_example": "{:.2f}", "tokens_per_sec": "{:.1f}", "n": "{:.0f}"})
        .background_gradient(cmap="Greens", subset=pct + ["queue_macro_f1", "priority_macro_f1", "rougeL_resolution"]))

# Save the evidence: run settings + data hashes (which frozen data was scored) + results + timestamp
summary = {"model": MODEL_ID, "load_4bit": LOAD_4BIT, "dtype": str(dtype), "max_new_tokens": MAX_NEW_TOKENS, "greedy": True,
           "gen_batch": GEN_BATCH, "k_shot": K_SHOT, "smoke_n": SMOKE_N,
           "data_sha256": json.loads((OUT / "manifest.json").read_text())["sha256"],
           "results": {m: {k: float(v) for k, v in r.items()} for m, r in rows.items()},
           "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
(OUT / f"baseline_results{'_smoke' if SMOKE_N else ''}.json").write_text(json.dumps(summary, indent=2))

if USE_WANDB:                                        # experiment tracking (capstone deliverable)
    import wandb
    for mode, r in rows.items():
        run = wandb.init(project="it-helpdesk-capstone", name=f"baseline_{mode}", config=summary, reinit=True)
        wandb.log({k: float(v) for k, v in r.items()}); run.finish()

,n,json_parse_rate,schema_valid_rate,queue_acc,type_acc,priority_acc,labels_exact_match,queue_macro_f1,priority_macro_f1,rougeL_resolution,mean_out_tokens,mean_prompt_len,sec_per_example,tokens_per_sec
zero_shot,180,100.0%,100.0%,65.0%,72.8%,50.0%,20.0%,0.494,0.379,0.220,91,254,0.34,270.2
few_shot,180,100.0%,100.0%,65.6%,71.1%,52.8%,23.9%,0.435,0.431,0.256,106,815,0.43,248.9


### Step 5a (continued) · Reading the results: what the baseline numbers say

Measured on the **180 held-out test tickets** with `Qwen2.5-7B-Instruct` in 4-bit (`bfloat16` compute) on an NVIDIA A100 40 GB, greedy decoding, `MAX_NEW_TOKENS = 320` (the numbers below are copied from `baseline_results.json` of this run; an earlier run on Tesla T4s in `float16` gave the same quality within noise but about 40% lower generation speed). These are the numbers Week 2 and Week 3 must beat.

| Metric | Zero-shot | Few-shot (k = 3) | Change |
|---|---|---|---|
| JSON parse rate | 100.0% | 100.0% | = |
| Schema-valid rate | 99.4% | 100.0% | +0.6 pp (1 ticket) |
| Queue accuracy | 63.9% | 65.6% | +1.7 pp (3 tickets) |
| Type accuracy | 71.7% | 71.1% | −0.6 pp |
| Priority accuracy | 50.6% | 52.2% | +1.6 pp |
| All three labels exact | 20.6% | 23.9% | +3.3 pp (6 tickets) |
| Queue macro-F1 | 0.484 | 0.435 | −0.049 |
| Priority macro-F1 | 0.383 | 0.425 | +0.042 |
| ROUGE-L of `resolution` | 0.219 | 0.260 | +0.041 |
| Mean prompt length (tokens) | 254 | 815 | 3.2× |
| Mean output length (tokens) | 91 | 105 | +16% |
| Generation speed (tokens/s) | 125.0 | 125.7 | = |
| Seconds per ticket | 0.73 | 0.84 | +15% |

**How to read these numbers**

1. **Output format is not the problem.** Every answer parsed as JSON and 179–180 of 180 were schema-valid. The single schema failure (zero-shot) invented a label that is not in the allowed list. This is the most important surprise of Week 1: with a strong instruct model and a clear system prompt, *format* is already solved, so fine-tuning cannot win much there. Any gain in Week 2 has to come from the **labels and the reply content**.
2. **Queue accuracy (63.9–65.6%) is no better than guessing the most common queue.** `Technical Support` is 116 of the 180 test tickets (64.4%), so a "model" that always answered `Technical Support` would score 64.4%, which is *above* the zero-shot result. The macro-F1 (0.484 / 0.435) shows the real picture: the classes are not being separated. **Always compare accuracy with the majority-class floor on an imbalanced label.**
3. **Priority accuracy (50.6–52.2%) is *below* always answering `high`**, which is 97 of the 180 test tickets (53.9%; about 56% in the whole filtered pool). The model's idea of urgency does not match the dataset's labels. These labels are dataset conventions, not something an instruct model can infer from the ticket text alone.
4. **Few-shot helps the reply style, not the judgement.** ROUGE-L rises from 0.219 to 0.260 (the examples show the target tone and structure) and priority macro-F1 rises, but queue macro-F1 *falls*. The three examples cannot teach where `IT Support` ends and `Technical Support` begins (see the error analysis below).
5. **Noise level.** With 180 tickets, the standard error of an accuracy near 65% is about **3.6 percentage points** (√(0.65 × 0.35 / 180)), so a 95% interval is roughly ±7 pp. Differences such as 63.9% vs 65.6%, or 71.7% vs 71.1%, are **indistinguishable from zero**; even the +3.3 pp in exact label match is only 6 tickets. Do not report a baseline difference of a few points as a finding.
6. **ROUGE-L of 0.22–0.26 is low, and that is expected.** The reference replies are paraphrastic, templated text; two good replies rarely share long word sequences. Use ROUGE-L to compare systems with each other, not as an absolute quality score, and add the LLM-as-judge in Week 3 as a second opinion.
7. **Cost.** The few-shot prompt is 3.2× longer than zero-shot. On the A100 the decode speed is unchanged (prompt processing is cheap there) but each ticket still takes 15% longer, mostly because the answers are longer; on a T4 the same prompt cost about 20% of the generation speed. Either way it buys a small, uneven gain. Remember this trade-off: Week 3's RAG prompts also add context, so their cost must be justified by a measurable gain.

**What this means for the next weeks**

- **Week 2 (fine-tuning)** must show gains in `queue_macro_f1`, `priority_macro_f1`, `labels_exact_match` and `rougeL_resolution`. Beating the zero-shot numbers above *and* the majority-class floor is the minimum bar; a result that only equals the floor suggests the labels are close to unlearnable and belongs in the limitations.
- **Week 3 (RAG)** targets exactly the weakness found here: retrieved, already-labelled similar tickets can show the model the dataset's labelling convention.

### Step 5b · Error analysis: *why* did the model lose points?

**What the next cell does**

A score tells you *how well* the model did; error analysis tells you *what went wrong*. This is where the "main failure modes of the base model" for the Week 1 checklist come from, and it tells you what fine-tuning should fix.

**For each baseline it prints three things**

1. **A failure breakdown**, counted from the per-row scores:
   - *unparseable JSON*: no JSON object could be found at all (format failure).
   - *parsed, schema invalid*: JSON was found but keys are missing/extra, or a label is not in the allowed list.
   - *valid, wrong queue* / *valid, wrong priority*: the format is fine but the routing or urgency decision is wrong. We only count these among schema-valid rows so format problems and judgement problems are not mixed.
   - *hit generation cap*: the model used all `MAX_NEW_TOKENS` tokens. A reply that gets cut off mid-sentence usually produces broken JSON, so a high number suggests raising the cap or asking for shorter replies.
2. **A confusion table** (`pd.crosstab`): rows are the *true* queue, columns the *predicted* queue. Numbers on the diagonal are correct; off-diagonal numbers show which queues the model mixes up. An `INVALID` column holds the answers with no usable queue.
3. **Up to two examples of invalid output**, truncated to 500 characters, so you can read with your own eyes what the model actually wrote (chatty preamble, markdown fences, unfinished JSON, and so on).

**Tip:** compare zero-shot and few-shot here. If few-shot cuts the format failures a lot, the examples are teaching the format; if the wrong-queue count stays high in both, that is a knowledge/labelling issue that fine-tuning (not prompting) needs to address.

In [14]:
for mode, d in scored_by_mode.items():
    print(f"=== {mode} ===")
    # Where did the points go? Format failures are separated from wrong-decision failures
    fail = pd.Series({
        "unparseable JSON": int((~d["parsed"]).sum()),
        "parsed, schema invalid": int((d["parsed"] & ~d["schema_valid"]).sum()),
        # wrong-label counts only use schema-valid rows, so a format error isn't double-counted as a wrong label
        "valid, wrong queue": int((d["schema_valid"] & ~d["queue_ok"]).sum()),
        "valid, wrong priority": int((d["schema_valid"] & ~d["priority_ok"]).sum()),
        # hitting the token cap usually means the answer was cut off mid-way (and the JSON is broken)
        "hit generation cap": int((d["gen_tokens"] >= MAX_NEW_TOKENS).sum()),
    })
    print(fail.to_string())
    # Confusion table: rows = true queue, columns = predicted queue; off-diagonal cells are the mix-ups
    print(pd.crosstab(gold["queue"].rename("gold"), d["pred_queue"].rename("pred")).to_string(), "\n")
    # Read a couple of broken outputs yourself: often the quickest way to spot the pattern (fences, chatter, cut-offs)
    for _, r in d[~d["schema_valid"]].head(2).iterrows():
        print("--- invalid output example ---\n", r["pred_text"][:500], "\n")

=== zero_shot ===
unparseable JSON           0
parsed, schema invalid     0
valid, wrong queue        63
valid, wrong priority     90
hit generation cap         0
pred                             IT Support  Service Outages and Maintenance  Technical Support
gold                                                                                           
IT Support                                6                                2                 40
Service Outages and Maintenance           3                                8                  5
Technical Support                         8                                5                103 

=== few_shot ===
unparseable JSON           0
parsed, schema invalid     0
valid, wrong queue        62
valid, wrong priority     85
hit generation cap         0
pred                             IT Support  Service Outages and Maintenance  Technical Support
gold                                                                                           
I

### Step 5b (continued) · Reading the error analysis: the model collapses onto the biggest queue

| Count (of 180 test tickets) | Zero-shot | Few-shot |
|---|---|---|
| Unparseable JSON | 0 | 0 |
| Parsed but schema-invalid | 1 | 0 |
| Valid, wrong queue | 65 | 62 |
| Valid, wrong priority | 88 | 86 |
| Hit the 320-token generation cap | 0 | 0 |

**Confusion tables turned into per-queue numbers** (true queue → how often the model found it)

| True queue (tickets) | Zero-shot recall | Few-shot recall | Zero-shot precision | Few-shot precision |
|---|---|---|---|---|
| Technical Support (116) | 101/116 = 87.1% | 108/116 = 93.1% | 101/147 = 68.7% | 108/157 = 68.8% |
| IT Support (48) | **6/48 = 12.5%** | **1/48 = 2.1%** | 6/17 = 35.3% | 1/1 = 100% |
| Service Outages and Maintenance (16) | 8/16 = 50.0% | 9/16 = 56.3% | 8/16 = 50.0% | 9/22 = 40.9% |

**What the error analysis tells us**

1. **The failure is judgement, not format.** No output was unparseable, none hit the length cap (so `MAX_NEW_TOKENS = 320` is generous: the mean answer is only 91–105 tokens), and only one answer was schema-invalid. The 62–65 wrong queues and 86–88 wrong priorities are *decisions*.
2. **The model sends almost everything to `Technical Support`.** It predicted that queue 147 times (82%) zero-shot and 157 times (87%) few-shot, against a true share of 64%. The recall of `IT Support` is 12.5% and drops to 2.1% with few-shot: 40 (zero-shot) and 42 (few-shot) of the 48 `IT Support` tickets were routed to `Technical Support`. That is a *label-convention* problem: from the ticket text alone, "IT Support" and "Technical Support" are nearly interchangeable, and the dataset's choice between them is not something a general model can guess.
3. **Why few-shot did not fix it.** The three worked examples (one per most-frequent queue) are far too few to show where `IT Support` ends and `Technical Support` begins, and queue macro-F1 actually fell from 0.484 to 0.435. `IT Support` precision of "100%" is misleading: the queue was predicted only once. Always read precision together with recall and the counts behind them.
4. **The one invalid output** (zero-shot) returned a `type` of `"Information Request"`, which is not one of the allowed values (`Change`, `Incident`, `Problem`, `Request`). It is a good illustration of why `schema_valid` is strict: a plausible-looking but invented label would silently corrupt a real ticketing system.
5. **What fine-tuning and retrieval need to fix:** teach the model the dataset's own boundary between `IT Support` and `Technical Support`, and its priority conventions. Fine-tuning does that through the weights (Week 2); retrieval does it by showing labelled neighbours at answer time (Week 3).

### Step 5c · Optional: LLM-as-judge (a second opinion on reply quality)

**What the next cell does**

ROUGE-L only measures *word overlap* with the reference reply. A reply can be perfectly good and still use different words, so overlap alone can under-rate it. This cell sketches a more human-like check: we ask a stronger "judge" model to read the ticket, the reference reply and our model's reply, then score the reply from 1 to 5 on four criteria (helpfulness, faithfulness, tone, safety).

**How it works, piece by piece**

1. `JUDGE_RUBRIC` is the grading instructions. It forces a JSON-only answer so we can parse the scores automatically.
2. `call_judge()` sends one prompt to the judge model through the Anthropic API.
3. If `RUN_JUDGE = True`, we loop over each baseline (zero-shot, few-shot), judge the first `JUDGE_N` rows, and average the scores per criterion.
4. If `RUN_JUDGE = False` (the default), nothing is called and we only print a reminder. **No API key is needed to run the notebook.**

**Good practice to remember:** use a judge from a *different model family* than the one that generated your training data. Models tend to favour text that looks like their own, and you should hand-check about 30 scores to make sure the judge agrees with your own eyes.

In [15]:
# Optional LLM-as-judge on a sample (Week 3 deliverable, previewed here). Off by default: needs an API key.
# The rubric is the "exam paper" the judge model sees: four criteria, each scored 1-5.
# Asking for JSON-only output lets us parse the scores with extract_json() instead of guessing at prose.
JUDGE_RUBRIC = """You grade an IT helpdesk assistant's reply. Score each criterion 1-5 (5 best).
- helpfulness: concrete, actionable next steps for this ticket
- faithfulness: no invented policies, tools, ticket IDs or credentials; consistent with the reference reply's intent
- tone: professional, empathetic, concise
- safety: does not request passwords, secrets or unsafe actions
Return only JSON: {"helpfulness": int, "faithfulness": int, "tone": int, "safety": int}"""

JUDGE_ERRORS = []                                    # collects why calls failed, so a wrong key is never silent

def call_judge(prompt, retries=4):
    """One judge call, with retries. Which service is used depends on the environment variables you set (never type a key into a notebook):
       - ANTHROPIC_API_KEY                      -> the Anthropic API (JUDGE_MODEL, default claude-sonnet-5-5)
       - JUDGE_API_KEY + JUDGE_BASE_URL         -> any OpenAI-compatible chat endpoint (also set JUDGE_MODEL to its model name)
       Returns None if every attempt fails."""
    for attempt in range(retries):
        try:
            if os.environ.get("ANTHROPIC_API_KEY"):
                import anthropic                              # imported here so the notebook runs without the package when the judge is off
                msg = anthropic.Anthropic().messages.create(model=JUDGE_MODEL, max_tokens=200, system=JUDGE_RUBRIC,
                                                            messages=[{"role": "user", "content": prompt}])
                return msg.content[0].text
            import requests
            r = requests.post(os.environ["JUDGE_BASE_URL"].rstrip("/") + "/chat/completions", timeout=120,
                              headers={"Authorization": "Bearer " + os.environ["JUDGE_API_KEY"]},
                              json={"model": JUDGE_MODEL, "temperature": 0, "max_tokens": 200,
                                    "messages": [{"role": "system", "content": JUDGE_RUBRIC}, {"role": "user", "content": prompt}]})
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"]
        except Exception as e:
            JUDGE_ERRORS.append(f"{type(e).__name__}: {str(e)[:200]}")
            time.sleep(2 ** attempt)                          # back off, then retry (rate limits are the usual cause)
    return None

def check_judged(name, n_judged, n_asked):
    """Fail loudly when the judge returned (almost) nothing, instead of printing an empty table."""
    print(f"{name}: judged {n_judged} of {n_asked} tickets")
    if n_judged == 0:
        raise RuntimeError("the judge returned no scores. First error seen: " + (JUDGE_ERRORS[0] if JUDGE_ERRORS else "none (check the replies)"))

if RUN_JUDGE:
    judge_rows = {}
    for mode, d in scored_by_mode.items():
        out = []
        # Judge only the first JUDGE_N rows: API calls cost money and a sample is enough to see the trend
        for g, (_, r) in list(zip(gold.to_dict("records"), d.iterrows()))[:JUDGE_N]:
            obj = extract_json(r["pred_text"]) or {}
            # The judge sees the ticket, the human reference reply and the model's reply together;
            # if the model produced no valid JSON, a placeholder lets the judge score it low
            prompt = (f"TICKET:\n{ticket_text(g)}\n\nREFERENCE REPLY:\n{g['answer']}\n\n"
                      f"MODEL REPLY:\n{obj.get('resolution', '(no valid reply)')}")
            s = extract_json(call_judge(prompt) or "")      # None (failed call) -> skipped below
            if s: out.append(s)                       # skip the rare case where the judge itself returned junk
        # average each criterion over the judged rows -> one small score card per baseline
        check_judged(mode, len(out), min(JUDGE_N or len(d), len(d)))
        judge_rows[mode] = pd.DataFrame(out).mean().round(2)
    display(pd.DataFrame(judge_rows).T)
else:
    print("RUN_JUDGE=False; skipping. Use a judge from a different model family than your teacher, and hand-check ~30 scores.")

RUN_JUDGE=False; skipping. Use a judge from a different model family than your teacher, and hand-check ~30 scores.


### Step 6 · Free the GPU memory (run this when you are finished)

**What the next cell does**

Everything above is saved to disk (`manifest.json`, `preds_*.jsonl`, `baseline_results.json`), so the 7B model sitting in GPU memory is no longer needed. But a notebook kernel keeps it there **even while idle**, and on a shared GPU that blocks your other notebooks. This is exactly what caused a `CUDA out of memory` error when the Week 2 training notebook was started while this one was still open. **Run this cell before you open the Week 2 notebook.**

**Step by step**

1. `gpu_report("before: ")` shows the memory in use right now.
2. We delete the Python variables that hold the model, and call `release_model()` on any module we find, which swaps its weights for empty tensors so the GPU storage is freed even if some library still references the module. GPU memory is only released once *nothing* refers to it any more, so calling `empty_cache()` on its own would free nothing. `globals().pop(name, None)` removes a variable if it exists and quietly does nothing if it does not, so the cell is safe to run twice.
3. `free_gpu()` (defined in Step 4) clears any saved error traceback, runs the garbage collector, and returns PyTorch's cached memory to the GPU driver.
4. `gpu_report("after:  ")` shows the result. Whatever is still used belongs to the CUDA runtime itself (roughly 0.3-0.5 GB) or to another program.
5. **Full shutdown (on by default).** Because `SHUTDOWN_WHEN_DONE = True` in the config cell, the cell ends by stopping the kernel process. That is the *only* way to return 100% of the GPU memory, because the CUDA runtime lives as long as the process does. The exit is scheduled by a 3-second timer so that **this cell finishes normally first** (exiting inside the cell would leave it showing "running" forever). About 3 seconds later VS Code reports the kernel as stopped or "died"; that is expected, and everything is already saved on disk. Re-run the notebook (or just open Week 2) when you need the GPU again. Set `SHUTDOWN_WHEN_DONE = False` to keep the kernel alive instead.

**Good to know**

- After this cell, the generation cells will fail because `model` is gone. Re-run *Load the base model* if you need it again.
- If memory stays high after this cell, something else still refers to the model. *Kernel > Restart* or *Shutdown* for this notebook always works.

In [16]:
gpu_report("before: ")

# Drop every global that may hold GPU tensors (pop() ignores names that don't exist, so re-running is safe)
for name in ["model", "kw", "out", "enc"]:
    obj = globals().pop(name, None)
    if isinstance(obj, torch.nn.Module):
        release_model(obj)                # free the weights' GPU storage even if something still references the module
free_gpu("after:  ")                      # clear tracebacks, garbage-collect, return cached memory to the driver

# Optional: end the process, because only a process exit releases the CUDA runtime's own ~0.3-0.5 GB
if SHUTDOWN_WHEN_DONE:
    import threading
    print("Shutting the kernel down in 3 seconds to free ALL GPU memory. Results are saved on disk; open Week 2 next.")
    sys.stdout.flush(); sys.stderr.flush()
    # exit from a timer thread, AFTER this cell has finished: exiting inside the cell would leave it "running" forever
    threading.Timer(3.0, os._exit, args=(0,)).start()

before: GPU 0: 5.7/44.4 GiB used | this kernel: 5.2 GiB allocated, 5.2 GiB reserved


after:  GPU 0: 0.5/44.4 GiB used | this kernel: 0.0 GiB allocated, 0.0 GiB reserved
Shutting the kernel down in 3 seconds to free ALL GPU memory. Results are saved on disk; open Week 2 next.


## 4 · Week 1 gate checklist

- [ ] `manifest.json` written and **committed**; `test.jsonl` never used for training, prompt selection or tuning
- [ ] Leakage guard ran (printed the number of dropped near-duplicates)
- [ ] Zero-shot and few-shot numbers recorded in `baseline_results.json` (and W&B/MLflow if `USE_WANDB`)
- [ ] Error analysis read, and the main failure modes of the base model written down
- [ ] Every team member has run this notebook (or a section of it) on the GPU box
- [ ] Step 6 run (GPU memory released) before starting the Week 2 notebook

**Measured results** (180 test tickets, Qwen2.5-7B-Instruct 4-bit, bfloat16 compute, NVIDIA A100 40 GB; copied from `baseline_results.json` and explained in the *Reading the results* cell above):

| Baseline | Schema valid | Queue macro-F1 | Priority macro-F1 | ROUGE-L | Tokens/s |
|---|---|---|---|---|---|
| zero-shot | 99.4% | 0.484 | 0.383 | 0.219 | 125.0 |
| few-shot ($k$ = 3) | 100.0% | 0.435 | 0.425 | 0.260 | 125.7 |

**Next (Week 2):** open `it_helpdesk_week2_qlora.ipynb`. It reuses `train.jsonl` / `val.jsonl` and the `messages` field for QLoRA, and it re-checks the SHA-256 hashes in `manifest.json` before it trains. Keep `SYSTEM`, decoding and `LOAD_4BIT` unchanged so the comparison stays clean. **Run Step 6 first** (or shut this kernel down) so Week 2 starts with an empty GPU.

**After Week 2:** open `it_helpdesk_week3_rag.ipynb` for the retrieval-augmented layer; it re-uses this notebook's `baseline_results.json` and `preds_zero_shot.jsonl` as its base-model reference.
